# Run Pi 0.5 on LIBERO — v2 Controlled Experiment

**Prerequisite:** v1 [`test_pi05_jennifer.ipynb`](test_pi05_jennifer.ipynb) completed (Phase 1/2/3 on Drive).

**Every session:** Section 1 → 2 → **3b** (upgrades torch if needed) → 4 → 5 → 6

> Package snapshot must exist at `smolvla_colab_cache/site_packages.tar.gz` (created by v1 Section 3).
> If missing, run Section 3 once (full install), then use 3b thereafter.

**Section 6:** imports v1 results, then runs only `extra_steps` + `multi_sample_select`.



---
## Section 1: Drive mount


In [1]:
from google.colab import drive
import os
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive'

# ── Shared project folder (confirm name with: !ls /content/drive/MyDrive)
SHARED = f'{DRIVE}/cs159-sp26'

# ── Section 3 snapshot: leave where Section 3 already wrote it
CACHE_DIR = f'{DRIVE}/smolvla_colab_cache'   # site_packages.tar.gz for 3b ONLY

# ── Everything you want collaborators to see
HF_HOME     = f'{DRIVE}/smolvla_colab_cache/hf_models' # UPDATE TO SHARED
RESULTS_DIR = f'{SHARED}/results_v2'

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(HF_HOME, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

os.environ['HF_HOME'] = HF_HOME
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['MUJOCO_GL'] = 'egl'

SNAPSHOT = f'{CACHE_DIR}/site_packages.tar.gz'
snapshot_exists = os.path.exists(SNAPSHOT)
print(f'Shared:     {SHARED}')
print(f'Cache dir:  {CACHE_DIR}')
print(f'HF models:  {HF_HOME}')
print(f'Results:    {RESULTS_DIR}')
if snapshot_exists:
    print('Snapshot:   FOUND — run Section 3b (skip Section 3)')
else:
    print('Snapshot:   NOT FOUND — run Section 3 once, then use 3b on future sessions')


Mounted at /content/drive
Shared:     /content/drive/MyDrive/cs159-sp26
Cache dir:  /content/drive/MyDrive/smolvla_colab_cache
HF models:  /content/drive/MyDrive/smolvla_colab_cache/hf_models
Results:    /content/drive/MyDrive/cs159-sp26/results_v2
Snapshot:   FOUND — run Section 3b (skip Section 3)


---
## Section 2: GPU check


In [2]:
import subprocess
out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print(out)
print('GPU detected above. torch is imported after Section 3b restore.')


NVIDIA A100-SXM4-40GB, 40960 MiB
GPU detected above. torch is imported after Section 3b restore.


---
## Section 3b: Fast restore (run every session)

Restores packages from Drive snapshot (~6 min). **Run this before Section 4.**


In [3]:
import subprocess, sys, os, time, shutil, importlib

SNAPSHOT = f'{CACHE_DIR}/site_packages.tar.gz'
LOCAL_SNAPSHOT = '/content/site_packages_restore.tar.gz'

if not os.path.exists(SNAPSHOT):
    raise FileNotFoundError(
        'No snapshot found. Run Section 3 (full install) first to create it.'
    )

t0 = time.time()
size_mb = os.path.getsize(SNAPSHOT) / 1e6
print(f'Found snapshot on Drive: {size_mb:.0f} MB')

print('Copying snapshot from Drive to local disk (avoids Drive timeout bugs)...')
shutil.copy(SNAPSHOT, LOCAL_SNAPSHOT)

print('Extracting packages...')
result = subprocess.run(
    ['tar', '-xzf', LOCAL_SNAPSHOT, '-C', '/'],
    capture_output=True, text=True
)
os.remove(LOCAL_SNAPSHOT) # Cleanup

if result.returncode != 0:
    print('Restore failed:', result.stderr[:500])
else:
    elapsed = time.time() - t0
    print(f'Restored in {elapsed:.1f}s')

# Force Python to recognize the newly extracted modules
importlib.invalidate_caches()

# ── Fix torch/diffusers mismatch (CUSTOM_KEY ImportError on Colab) ───────────
# Colab's pre-installed torch is often <2.6; restored diffusers expects
# torch.ao.quantization.CUSTOM_KEY. Upgrade torch before any lerobot import.
def _fix_torch_diffusers_compat():
    import subprocess, sys
    try:
        import torch
        from torch.ao.quantization import CUSTOM_KEY  # noqa: F401
        print(f"torch {torch.__version__} — diffusers compat OK")
        return
    except ImportError:
        print("torch/diffusers mismatch detected — upgrading torch...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "-U",
        "torch", "torchvision", "torchaudio",
    ])
    importlib.invalidate_caches()
    # Purge stale torch if Section 2 already imported an older build
    import sys
    for mod in list(sys.modules):
        if mod == "torch" or mod.startswith("torch."):
            del sys.modules[mod]
    import torch
    from torch.ao.quantization import CUSTOM_KEY  # noqa: F401
    print(f"torch upgraded to {torch.__version__} — diffusers compat OK")

_fix_torch_diffusers_compat()


# Quick check
for pkg in ['mujoco', 'robosuite', 'libero', 'lerobot']:
    try:
        importlib.import_module(pkg)
        print(f'  {pkg} OK')
    except ImportError as e:
        print(f'  {pkg} MISSING: {e}')
print('Restore complete.')

# Verify restored packages (must pass before Section 4)
import torch
from lerobot.policies.pi05.modeling_pi05 import PI05Policy
print('PI05 import: OK')
from libero.libero import benchmark
print('LIBERO import: OK')
import mujoco
print(f'MuJoCo: {mujoco.__version__}')
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')



Found snapshot on Drive: 11202 MB
Copying snapshot from Drive to local disk (avoids Drive timeout bugs)...
Extracting packages...
Restored in 352.0s


[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
[robosuite WARNING] To setup, run: python /usr/local/lib/python3.12/dist-packages/robosuite/scripts/setup_macros.py (__init__.py:9)


torch 2.10.0+cu128 — diffusers compat OK
  mujoco OK
  robosuite OK
  libero OK
  lerobot OK
Restore complete.
PI05 import: OK
Do you want to specify a custom path for the dataset folder? (Y/N): N
Initializing the default config file...
The following information is stored in the config file: /root/.libero/config.yaml
benchmark_root: /usr/local/lib/python3.12/dist-packages/libero/libero
bddl_files: /usr/local/lib/python3.12/dist-packages/libero/libero/./bddl_files
init_states: /usr/local/lib/python3.12/dist-packages/libero/libero/./init_files
datasets: /usr/local/lib/python3.12/dist-packages/libero/libero/../datasets
assets: /usr/local/lib/python3.12/dist-packages/libero/libero/./assets
LIBERO import: OK
MuJoCo: 3.8.1
Device: cuda


---
## Section 3: Full install (FIRST SESSION ONLY)

Skip if snapshot already exists from v1. At end, saves snapshot for 3b.


In [ ]:
%%bash
apt-get update -qq
apt-get install -y -qq libosmesa6-dev libgl1-mesa-glx libglfw3 libglew-dev libegl1-mesa-dev patchelf ffmpeg
# EGL fix for Colab
mkdir -p /usr/share/glvnd/egl_vendor.d
echo '{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}' \
    > /usr/share/glvnd/egl_vendor.d/10_nvidia.json
echo 'System deps done'


In [ ]:
!pip install \
    mujoco robosuite libero sentencepiece tiktoken \
    "transformers==5.5.4" \
    "lerobot[pi0] @ git+https://github.com/huggingface/lerobot.git@01dcb4c29222bc9f2388cebf87f0e79965a9508b" \
    "diffusers==0.30.2"



In [ ]:
# Verify packages + fix torch/diffusers compat
import subprocess, sys, importlib
import torch

try:
    from torch.ao.quantization import CUSTOM_KEY  # noqa: F401
except ImportError:
    print('Upgrading torch for diffusers compatibility...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'torch', 'torchvision', 'torchaudio'])
    importlib.invalidate_caches()
    import torch
print(f'torch {torch.__version__}')

from lerobot.policies.pi05.modeling_pi05 import PI05Policy
print("PI05 import: OK")

from libero.libero import benchmark
print("LIBERO import: OK")

import mujoco
print(f"MuJoCo: {mujoco.__version__}")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")



In [ ]:
# Login + cache pi0.5 checkpoint (Section 3 only — skip if v1 already cached HF weights)
from huggingface_hub import login, snapshot_download
import os

hf_token = os.getenv('HF_TOKEN')
login(token=hf_token) if hf_token else login()

print(f'Downloading pi0.5 LIBERO checkpoint to {os.environ["HF_HOME"]} ...')
path = snapshot_download(repo_id='lerobot/pi05_libero_finetuned')
print(f'Checkpoint cached at: {path}')
# Tokenizer is loaded internally by lerobot preprocess — do NOT import transformers here.


In [ ]:
# Save snapshot of installed packages to Drive
# This is the step that makes future sessions fast (~2-3 min restore vs 15-20 min reinstall)
import subprocess, os, sys, shutil

print('Saving package snapshot locally first, then moving to Drive...')
LOCAL_SNAPSHOT = '/content/site_packages.tar.gz'
DRIVE_SNAPSHOT = f'{CACHE_DIR}/site_packages.tar.gz'

# Save site-packages automatically detecting python version
site_packages_dir = f'usr/local/lib/python3.{sys.version_info.minor}/dist-packages'

# 1. Tar to local disk (very fast)
result = subprocess.run([
    'tar', '-czf', LOCAL_SNAPSHOT,
    '-C', '/',
    site_packages_dir,
    'usr/local/bin',       # includes lerobot-train, lerobot-eval etc.
], capture_output=True, text=True)

if result.returncode == 0:
    # 2. Copy to Drive
    print('Copying to Google Drive...')
    shutil.copy(LOCAL_SNAPSHOT, DRIVE_SNAPSHOT)
    os.remove(LOCAL_SNAPSHOT) # clean up local copy

    size_mb = os.path.getsize(DRIVE_SNAPSHOT) / 1e6
    print(f'Snapshot saved: {DRIVE_SNAPSHOT} ({size_mb:.0f} MB)')
    print('Future sessions can use Section 3b (fast restore)')
else:
    print('Warning: snapshot failed. Check error:')
    print(result.stderr[:500])


---
## Section 4: Load policy + LIBERO helpers


In [4]:
# Guard: if 3b was skipped, fix torch/diffusers here
try:
    from torch.ao.quantization import CUSTOM_KEY  # noqa: F401
except ImportError:
    import subprocess, sys, importlib
    print('Upgrading torch (run Section 3b first in future sessions)...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'torch', 'torchvision', 'torchaudio'])
    importlib.invalidate_caches()
    import sys
    for mod in list(sys.modules):
        if mod == 'torch' or mod.startswith('torch.'):
            del sys.modules[mod]

# ── Section 4: Load policy + LIBERO helpers ───────────────────────────────
import os, time, json, math
import torch, numpy as np
from tqdm.notebook import tqdm
from huggingface_hub import login
from libero.libero import benchmark, get_libero_path
from libero.libero.envs import OffScreenRenderEnv
from lerobot.policies.pi05.modeling_pi05 import PI05Policy
from lerobot.policies.factory import make_pre_post_processors

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

hf_token = os.getenv('HF_TOKEN')
if hf_token:
    login(token=hf_token)
else:
    login()

policy = PI05Policy.from_pretrained('lerobot/pi05_libero_finetuned').to(device).eval()
print(f'PI05 loaded on {device}. Params: {sum(p.numel() for p in policy.parameters())/1e6:.0f}M')

MAX_STEPS_MAP = {
    'libero_spatial': 220,
    'libero_object':  280,
    'libero_goal':    300,
    'libero_10':      520,
    'libero_90':      400,
}

CAMERAS = ['agentview', 'robot0_eye_in_hand']
IMG_SIZE = 360
LIBERO_DUMMY_ACTION = [0.0] * 6 + [-1.0]
NUM_STEPS_WAIT = 10


def _quat2axisangle(quat):
    if quat[3] > 1.0:  quat[3] = 1.0
    elif quat[3] < -1.0: quat[3] = -1.0
    den = np.sqrt(1.0 - quat[3] * quat[3])
    if math.isclose(den, 0.0):
        return np.zeros(3)
    return (quat[:3] * 2.0 * math.acos(quat[3])) / den


preprocess, postprocess = make_pre_post_processors(
    policy.config,
    'lerobot/pi05_libero_finetuned',
    preprocessor_overrides={'device_processor': {'device': str(device)}},
)


def obs_to_policy(obs_dict, task_desc, device):
    agentview = np.ascontiguousarray(obs_dict['agentview_image'][::-1, ::-1])
    wrist     = np.ascontiguousarray(obs_dict['robot0_eye_in_hand_image'][::-1, ::-1])
    img_agent = torch.from_numpy(agentview / 255.0).permute(2, 0, 1).float()
    img_wrist = torch.from_numpy(wrist / 255.0).permute(2, 0, 1).float()
    state = np.concatenate([
        obs_dict['robot0_eef_pos'],
        _quat2axisangle(obs_dict['robot0_eef_quat']),
        obs_dict['robot0_gripper_qpos'],
    ])
    return {
        'observation.images.image':  img_agent,
        'observation.images.image2': img_wrist,
        'observation.state': torch.from_numpy(state).float(),
        'task': task_desc,
    }

benchmark_dict = benchmark.get_benchmark_dict()
print('Section 4 ready.')




/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


The PI05 model is a direct port of the OpenPI implementation. 
This implementation follows the original OpenPI structure for compatibility. 
Original implementation: https://github.com/Physical-Intelligence/openpi


/usr/local/lib/python3.12/dist-packages/torch/jit/_script.py:362: DeprecationWarning: `torch.jit.script_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Loading model from: lerobot/pi05_libero_finetuned


model.safetensors:   0%|          | 0.00/7.47G [00:00<?, ?B/s]

✓ Loaded state dict from model.safetensors
All keys loaded successfully!
PI05 loaded on cuda. Params: 4143M
Section 4 ready.


---
## Section 5: P&P sampler + RolloutDB


In [5]:
# ── Predict-and-Perturb (P&P) sampler: config, recorder, patched Euler loop ──
import torch, numpy as np
from dataclasses import dataclass
from typing import Optional, Sequence
from lerobot.policies.pi05.modeling_pi05 import make_att_2d_masks


@dataclass
class PnPConfig:
    """Self-refining (Predict-and-Perturb) sampling config for Pi0.5 flow matching.

    LeRobot time runs s=1.0 (noise) -> s=0.0 (clean), so the paper's early/high-noise
    steps are the FIRST Euler steps (large s). Select them with step_indices=(1,)/(1,2)
    or time_min=0.8.

    NOTE on step 0 (s=1.0): the perturb (1-s)*a_hat + s*eps drops a_hat entirely and returns
    fresh noise. So REFINEMENT at s=1.0 is a no-op-like reseed (the map x->eps has no
    contraction). But UNCERTAINTY at s=1.0 is meaningful: each iteration predicts a_hat from
    an independent noise draw, so U measures the spread of the policy's one-shot action
    prediction over the noise prior (overall predictive variance given the observation).
    Default selection starts at step 1 so the *refinement* demo is non-trivial; step 0 is fine
    and informative for mode="uncertainty".
    """
    enabled: bool = False
    step_indices: Optional[Sequence[int]] = (1,)   # which Euler steps run P&P (ignored if time_min set)
    time_min: Optional[float] = None               # alt selector: run P&P when s >= time_min
    num_iterations: int = 3                         # K predict-and-perturb iterations
    mode: str = "both"                              # "uncertainty" | "refine" | "both"
    action_dim: int = 7                             # real (un-padded) action dims used for uncertainty
    record_per_iteration: bool = False              # also store the full (K,B,chunk,adim) a_hat stack per step

    def step_selected(self, step: int, s: float) -> bool:
        if not self.enabled:
            return False
        if self.time_min is not None:
            return s >= self.time_min
        return self.step_indices is not None and step in tuple(self.step_indices)

    @property
    def do_refine(self) -> bool:
        return self.mode in ("refine", "both")


class PnPRecorder:
    """Collects per-episode P&P uncertainty so it can later be correlated with outcomes.

    After a run, `episodes` is a list of dicts:
        {"meta": {...}, "success": bool, "n_steps": int,
         "chunks": [ {"num_steps": int,
                      "steps": [ {"step": i, "s": float,
                                  "u_consecutive": np[B,chunk,adim],  # Eq.10 mean|Δâ|
                                  "a_std": np[B,chunk,adim],          # spread of â over iters
                                  "u_mean", "u_max", "a_std_mean": float,
                                  "u_vec": np[adim],     # per-action-dim mean of u_consecutive
                                  "a_std_vec": np[adim], # per-action-dim mean of a_std
                                  # only if cfg.record_per_iteration:
                                  "a_hats": np[K,B,chunk,adim]}, ... ]}, ... ]}
    One "chunk" == one full action-chunk prediction (one sample_actions call).
    Action dims (LIBERO): 0-2 = xyz pos, 3-5 = axis-angle rot, 6 = gripper.
    """
    def __init__(self):
        self.reset()

    def reset(self):
        self.episodes = []
        self._cur = None

    def new_episode(self, meta=None):
        self._cur = {"meta": dict(meta or {}), "chunks": [], "success": None, "n_steps": None}

    def log_chunk(self, chunk_rec):
        if self._cur is not None:
            self._cur["chunks"].append(chunk_rec)

    def close_episode(self, success, n_steps):
        if self._cur is None:
            return
        self._cur["success"] = bool(success)
        self._cur["n_steps"] = int(n_steps)
        self.episodes.append(self._cur)
        self._cur = None


# Global handles (notebook-style); the patched method reads these each call.
PNP_CONFIG = PnPConfig()
PNP_RECORDER = PnPRecorder()

# v2: optional inference-step override for matched-compute baselines
INFERENCE_NUM_STEPS_OVERRIDE = None


_pnp_disable_compile = getattr(getattr(torch, "compiler", None), "disable", lambda fn: fn)


@_pnp_disable_compile
def _pnp_mark_cuda_graph_step():
    """Tell torch.compile/CUDA graphs that a new policy invocation is starting."""
    mark_step = getattr(getattr(torch, "compiler", None), "cudagraph_mark_step_begin", None)
    if mark_step is not None and torch.cuda.is_available():
        mark_step()


@_pnp_disable_compile
def _pnp_compile_mode(config):
    """Map LeRobot compile modes to CUDA-graph-safe equivalents for P&P sampling."""
    mode = getattr(config, "compile_mode", "default")
    if mode == "max-autotune":
        return "max-autotune-no-cudagraphs"
    return mode


@_pnp_disable_compile
def _pnp_log_chunk(chunk_rec):
    """Side-effect logging must stay outside torch.compile/CUDA graphs."""
    PNP_RECORDER.log_chunk(chunk_rec)


@_pnp_disable_compile
def _pnp_measure_only_actions(model, images, img_masks, tokens, masks, noise, num_steps, kwargs):
    """Run the saved original sampler for uncertainty-only (non-invasive) mode."""
    return model._orig_sample_actions(
        images, img_masks, tokens, masks, noise=noise, num_steps=num_steps, **kwargs
    ).clone()


@_pnp_disable_compile
def _pnp_refine_at_step(x_t, s, vfield, cfg):
    """Run K predict-and-perturb iterations at fixed noise level s.

        predict:  a_hat = x - s * v(x, s)
        perturb:  x'    = (1 - s) * a_hat + s * eps,   eps ~ N(0, I)

    Returns (x_out, rec). x_out is the refined re-noised state if cfg.do_refine, else
    the original x_t unchanged (uncertainty-only is non-invasive). `rec` always holds the
    uncertainty measured across iterations (a free by-product of the predicts).

    This probe does CPU/NumPy logging, so keep it out of torch.compile/CUDA graphs.
    Otherwise graph partitioning and static-buffer reuse can make "uncertainty" mode
    perturb the caller even though it returns the original x_t.
    """
    adim = cfg.action_dim
    x_acc = x_t
    a_hats = []
    for _ in range(cfg.num_iterations):
        v = vfield(x_acc)
        a_hat = x_acc - s * v                       # predicted clean action
        a_hats.append(a_hat[..., :adim])
        eps = torch.randn_like(x_acc)
        x_acc = (1.0 - s) * a_hat + s * eps         # perturb back to level s

    A = torch.stack(a_hats, dim=0)                  # (K, B, chunk, adim)
    if A.shape[0] >= 2:
        u_consecutive = (A[1:] - A[:-1]).abs().mean(dim=0)   # (B, chunk, adim)
        a_std = A.std(dim=0)                                  # (B, chunk, adim)
    else:
        u_consecutive = torch.zeros_like(A[0])
        a_std = torch.zeros_like(A[0])

    # Per-action-dim vectors: mean over batch and chunk → shape (adim,)
    u_vec     = u_consecutive.mean(dim=(0, 1)).detach().float().cpu().numpy()
    a_std_vec = a_std.mean(dim=(0, 1)).detach().float().cpu().numpy()

    rec = {
        "s":             float(s),
        "u_consecutive": u_consecutive.detach().float().cpu().numpy(),
        "a_std":         a_std.detach().float().cpu().numpy(),
        "u_mean":        float(u_consecutive.mean()),
        "u_max":         float(u_consecutive.max()),
        "a_std_mean":    float(a_std.mean()),
        "u_vec":         u_vec,      # np (adim,) — per-dim mean uncertainty
        "a_std_vec":     a_std_vec,  # np (adim,) — per-dim std of predictions
    }
    if cfg.record_per_iteration:
        rec["a_hats"] = A.detach().float().cpu().numpy()
    return (x_acc if cfg.do_refine else x_t), rec


@torch.no_grad()
def _sample_actions_pnp(self, images, img_masks, tokens, masks, noise=None, num_steps=None, **kwargs):
    """Drop-in replacement for PI05Pytorch.sample_actions with optional P&P refinement.

    Delegates to the saved original when P&P is disabled or under RTC; otherwise replicates
    the Euler loop verbatim and injects the P&P inner loop at the selected steps.

    CUDA-graph marks and recorder/logging live in @_pnp_disable_compile helpers *outside*
    this hot path so torch.compile can capture the Euler loop identically to the original.
    Call _pnp_mark_cuda_graph_step() once before each policy invocation (equivalence test,
    rollout, etc.) — not from inside here.
    """
    cfg = PNP_CONFIG
    if (not cfg.enabled) or self._rtc_enabled():
        return self._orig_sample_actions(
            images, img_masks, tokens, masks, noise=noise, num_steps=num_steps, **kwargs)

    if num_steps is None:
        num_steps = INFERENCE_NUM_STEPS_OVERRIDE or self.config.num_inference_steps
    bsize = tokens.shape[0]
    device = tokens.device
    if noise is None:
        actions_shape = (bsize, self.config.chunk_size, self.config.max_action_dim)
        noise = self.sample_noise(actions_shape, device)

    measure_only_output = None
    if cfg.mode == "uncertainty":
        # Guarantee non-invasive behavior: the returned action comes from the saved
        # original sampler, while the custom loop below only populates PNP_RECORDER.
        measure_only_output = _pnp_measure_only_actions(
            self, images, img_masks, tokens, masks, noise.clone(), num_steps, kwargs)

    # ---- prefix / KV cache: replicated verbatim from the original sample_actions ----
    prefix_embs, prefix_pad_masks, prefix_att_masks = self.embed_prefix(images, img_masks, tokens, masks)
    prefix_att_2d_masks = make_att_2d_masks(prefix_pad_masks, prefix_att_masks)
    prefix_position_ids = torch.cumsum(prefix_pad_masks, dim=1) - 1
    prefix_att_2d_masks_4d = self._prepare_attention_masks_4d(prefix_att_2d_masks)
    self.paligemma_with_expert.paligemma.model.language_model.config._attn_implementation = "eager"
    _, past_key_values = self.paligemma_with_expert.forward(
        attention_mask=prefix_att_2d_masks_4d,
        position_ids=prefix_position_ids,
        past_key_values=None,
        inputs_embeds=[prefix_embs, None],
        use_cache=True,
    )

    dt = -1.0 / num_steps
    x_t = noise
    chunk_rec = {"num_steps": num_steps, "steps": []}

    for step in range(num_steps):
        time = 1.0 + step * dt
        s = time
        time_tensor = torch.tensor(time, dtype=torch.float32, device=device).expand(bsize)

        def denoise_step_partial_call(input_x_t, current_timestep=time_tensor):
            return self.denoise_step(
                prefix_pad_masks=prefix_pad_masks,
                past_key_values=past_key_values,
                x_t=input_x_t,
                timestep=current_timestep,
            )

        if cfg.step_selected(step, s):
            x_t, rec = _pnp_refine_at_step(x_t, s, denoise_step_partial_call, cfg)
            rec["step"] = step
            chunk_rec["steps"].append(rec)

        v_t = denoise_step_partial_call(x_t)
        x_t = x_t + dt * v_t

    _pnp_log_chunk(chunk_rec)
    return measure_only_output if measure_only_output is not None else x_t

print("PnP defined: PnPConfig, PnPRecorder, PNP_CONFIG, PNP_RECORDER, _sample_actions_pnp")


PnP defined: PnPConfig, PnPRecorder, PNP_CONFIG, PNP_RECORDER, _sample_actions_pnp


In [6]:
# ── Apply the monkey-patch onto policy.model ──────────────────────────────
import types


def _infer_action_dim(policy, default=7):
    """Real (un-padded) action dim; LIBERO finetune uses 7."""
    feats = getattr(policy.config, "output_features", {})
    if "action" in feats:
        return int(feats["action"].shape[0])
    for mod in ("lerobot.utils.constants", "lerobot.constants"):
        try:
            mod_obj = __import__(mod, fromlist=["ACTION"])
            return int(feats[mod_obj.ACTION].shape[0])
        except Exception:
            continue
    return default


PNP_CONFIG.action_dim = _infer_action_dim(policy)
print(f"action_dim = {PNP_CONFIG.action_dim} | num_inference_steps = {policy.config.num_inference_steps}")

# Save original once; always re-patch from the saved original so this cell is idempotent.
if not hasattr(policy.model, "_orig_sample_actions"):
    policy.model._orig_sample_actions = policy.model.sample_actions
else:
    policy.model.sample_actions = policy.model._orig_sample_actions
policy.model.sample_actions = types.MethodType(_sample_actions_pnp, policy.model)

# PI05Pytorch.__init__ already compiled sample_actions. Recompile BOTH the saved
# original and the patched wrapper with the same CUDA-graph-safe mode so the
# delegated path (A) and reimplemented loop (B) stay numerically aligned.
if getattr(policy.model.config, "compile_model", False):
    compile_mode = _pnp_compile_mode(policy.model.config)

    def _unwrap_compiled(fn):
        while hasattr(fn, "_orig_mod"):
            fn = fn._orig_mod
        return fn

    policy.model._orig_sample_actions = torch.compile(
        _unwrap_compiled(policy.model._orig_sample_actions),
        mode=compile_mode,
    )
    policy.model.sample_actions = torch.compile(
        policy.model.sample_actions,
        mode=compile_mode,
    )
    print(f"Recompiled _orig_sample_actions + patched sample_actions (mode={compile_mode!r}).")

# Default to OFF so the rest of the notebook behaves exactly as before until you opt in.
PNP_CONFIG.enabled = False
print("Patched policy.model.sample_actions (PnP currently disabled).")


action_dim = 7 | num_inference_steps = 10
Recompiled _orig_sample_actions + patched sample_actions (mode='max-autotune-no-cudagraphs').
Patched policy.model.sample_actions (PnP currently disabled).


In [7]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
torch.use_deterministic_algorithms(True)

# ── Equivalence test: reimplemented loop == original when P&P is off ───────
# (A) PnP disabled -> delegates to the saved original loop.
# (B) PnP enabled but NO step selected -> exercises OUR reimplemented loop; must match (A)
#     bit-for-bit given the same fixed noise. (C) sanity: refinement actually changes output.
_EQUIV_SEED = 0

def _reset_equiv_seed():
    torch.manual_seed(_EQUIV_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(_EQUIV_SEED)

_equiv_suite = benchmark_dict['libero_spatial']()
task = _equiv_suite.get_task(0)
bddl = os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file)
init_states = _equiv_suite.get_task_init_states(0)

_env = OffScreenRenderEnv(
    bddl_file_name=bddl, camera_names=CAMERAS,
    camera_heights=IMG_SIZE, camera_widths=IMG_SIZE,
    has_offscreen_renderer=True, use_camera_obs=True,
    has_renderer=False, reward_shaping=False,
)
try:
    _env.reset(); policy.reset()
    obs = _env.set_init_state(init_states[0])
    for _ in range(NUM_STEPS_WAIT):
        obs, _, _, _ = _env.step(LIBERO_DUMMY_ACTION)
    batch = preprocess(obs_to_policy(obs, task.language, device))

    _reset_equiv_seed()
    fixed_noise = policy.model.sample_noise(
        (1, policy.config.chunk_size, policy.config.max_action_dim), device)
    print(f"[equiv seed] torch+cuda manual_seed({_EQUIV_SEED}); shared fixed_noise for (A)/(B)/(B2)")

    # (A) original loop (delegation)
    PNP_CONFIG.enabled = False
    _reset_equiv_seed()
    with torch.no_grad():
        _pnp_mark_cuda_graph_step()
        a_orig = policy.predict_action_chunk(batch, noise=fixed_noise.clone()).clone()

    # (B) our loop, P&P enabled but no step selected
    PNP_CONFIG.enabled = True
    PNP_CONFIG.step_indices = ()
    PNP_CONFIG.time_min = None
    _reset_equiv_seed()
    with torch.no_grad():
        _pnp_mark_cuda_graph_step()
        a_loop = policy.predict_action_chunk(batch, noise=fixed_noise.clone()).clone()

    max_abs = (a_orig - a_loop).abs().max().item()
    print(f"[loop-equivalence] max|Δ| = {max_abs:.3e}  ->", "OK" if max_abs < 1e-4 else "MISMATCH")
    assert torch.allclose(a_orig, a_loop, atol=1e-4), "Reimplemented loop diverged from original!"

    # (B2) P&P enabled WITH steps selected, but in measure-only mode -> non-invasive.
    #      The inner loop runs (and records uncertainty) yet must NOT change x_t, so the
    #      final action must still equal the original.
    PNP_CONFIG.step_indices = (1, 2)
    PNP_CONFIG.mode = "uncertainty"
    PNP_CONFIG.num_iterations = 3
    _reset_equiv_seed()
    with torch.no_grad():
        _pnp_mark_cuda_graph_step()
        a_meas = policy.predict_action_chunk(batch, noise=fixed_noise.clone()).clone()
    max_abs2 = (a_orig - a_meas).abs().max().item()
    print(f"[non-invasive]     max|Δ| = {max_abs2:.3e}  ->", "OK" if max_abs2 < 1e-4 else "MISMATCH")
    assert torch.allclose(a_orig, a_meas, atol=1e-4), "uncertainty-only mode changed the output!"

    # (C) refinement-on sanity: output should actually move
    PNP_CONFIG.step_indices = (1,)
    PNP_CONFIG.mode = "refine"
    PNP_CONFIG.num_iterations = 3
    _reset_equiv_seed()
    with torch.no_grad():
        _pnp_mark_cuda_graph_step()
        a_ref = policy.predict_action_chunk(batch, noise=fixed_noise.clone()).clone()
    print(f"[refine sanity]   max|Δ vs orig| = {(a_ref - a_orig).abs().max().item():.3e}  (expected > 0)")
finally:
    _env.close()
    PNP_CONFIG.enabled = False   # leave OFF by default
    PNP_CONFIG.step_indices = (1,)
    PNP_CONFIG.mode = "both"
print("Equivalence test passed; PnP left disabled.")


[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Local assets not found. Downloading from HuggingFace Hub...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 586 files:   0%|          | 0/586 [00:00<?, ?it/s]

Assets downloaded successfully to /root/.cache/libero/assets
[equiv seed] torch+cuda manual_seed(0); shared fixed_noise for (A)/(B)/(B2)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[loop-equivalence] max|Δ| = 0.000e+00  -> OK


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[non-invasive]     max|Δ| = 0.000e+00  -> OK


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


[refine sanity]   max|Δ vs orig| = 1.217e+00  (expected > 0)
Equivalence test passed; PnP left disabled.


In [17]:
# ── Recording-aware rollout + action-instability metrics (v2) ─────────────
import os, imageio

VIDEO_FPS = 10
if 'VIDEO_DIR' not in dir():
    _results = RESULTS_DIR if 'RESULTS_DIR' in dir() else '/content/drive/MyDrive/cs159-sp26/results_v2'
    VIDEO_DIR = os.path.join(_results, 'videos_v2')


def _agentview_frame(obs):
    return np.ascontiguousarray(obs['agentview_image'][::-1, ::-1])


def _compute_chunk_disagreement(chunk_boundary_actions):
    if len(chunk_boundary_actions) < 2:
        return None
    disagreements = [
        float(np.linalg.norm(chunk_boundary_actions[i + 1] - chunk_boundary_actions[i]))
        for i in range(len(chunk_boundary_actions) - 1)
    ]
    return float(np.mean(disagreements))


def _compute_action_instability(executed_actions, chunk_boundary_actions=None, gripper_dim=6, gripper_thresh=0.0):
    if not executed_actions:
        return {
            'action_delta_l2_mean': 0.0,
            'action_delta_l2_max': 0.0,
            'action_var_mean': 0.0,
            'gripper_flip_count': 0,
            'gripper_flip_rate': 0.0,
            'chunk_disagreement_mean': None,
        }
    arr = np.stack([np.asarray(a).flatten()[:getattr(PNP_CONFIG, 'action_dim', 7)] for a in executed_actions])
    if len(arr) >= 2:
        deltas = np.linalg.norm(np.diff(arr, axis=0), axis=1)
        action_delta_l2_mean = float(np.mean(deltas))
        action_delta_l2_max = float(np.max(deltas))
    else:
        action_delta_l2_mean = 0.0
        action_delta_l2_max = 0.0
    action_var_mean = float(np.var(arr, axis=0).mean())
    gripper = arr[:, gripper_dim]
    signs = (gripper > gripper_thresh).astype(int)
    gripper_flip_count = int(np.sum(np.diff(signs) != 0)) if len(signs) > 1 else 0
    gripper_flip_rate = gripper_flip_count / max(len(arr) - 1, 1)
    return {
        'action_delta_l2_mean': action_delta_l2_mean,
        'action_delta_l2_max': action_delta_l2_max,
        'action_var_mean': action_var_mean,
        'gripper_flip_count': gripper_flip_count,
        'gripper_flip_rate': gripper_flip_rate,
        'chunk_disagreement_mean': _compute_chunk_disagreement(chunk_boundary_actions or []),
    }


def _episode_seed(init_state, episode_idx):
    import hashlib as _hs
    _seed_bytes = _hs.md5(
        np.asarray(init_state).tobytes() + str(episode_idx or 0).encode()
    ).digest()
    return int.from_bytes(_seed_bytes[:4], 'big')


def _to_numpy_action(action):
    action = postprocess(action)
    if isinstance(action, torch.Tensor):
        action = action.squeeze(0).cpu().numpy()
    return np.asarray(action).flatten()


def run_episode_pnp(env, init_state, policy, task_desc, max_steps, device,
                    ep_meta=None, db=None,
                    suite=None, task_idx=None, episode_idx=None,
                    save_video=False,
                    method=None, final_eval_slice=0,
                    num_inference_steps=None, num_samples=None):
    env.reset(); policy.reset()
    obs = env.set_init_state(init_state)
    for _ in range(NUM_STEPS_WAIT):
        obs, _, _, _ = env.step(LIBERO_DUMMY_ACTION)

    _seed = _episode_seed(init_state, episode_idx)
    torch.manual_seed(_seed)
    torch.cuda.manual_seed(_seed)

    global INFERENCE_NUM_STEPS_OVERRIDE
    _prev_override = INFERENCE_NUM_STEPS_OVERRIDE
    if num_inference_steps is not None:
        INFERENCE_NUM_STEPS_OVERRIDE = num_inference_steps

    PNP_RECORDER.new_episode(ep_meta)
    record_video = save_video in (True, 'failures_only')
    frames = [] if record_video else None
    executed_actions = []
    chunk_boundary_actions = []
    last_n_chunks = 0

    rollout_id = None
    if suite is not None:
        rollout_id = RolloutDB.make_rollout_id(
            suite, task_idx or 0, episode_idx or 0, init_state, PNP_CONFIG,
            method=method, num_inference_steps=num_inference_steps, num_samples=num_samples)

    t0 = time.time(); success = False; step = 0
    try:
        for step in range(max_steps):
            if record_video:
                frames.append(_agentview_frame(obs))
            raw_obs = obs_to_policy(obs, task_desc, device)
            batch = preprocess(raw_obs)
            _pnp_mark_cuda_graph_step()
            with torch.no_grad():
                action = policy.select_action(batch)
            action_np = _to_numpy_action(action)
            executed_actions.append(action_np.copy())
            n_chunks = len(PNP_RECORDER._cur['chunks']) if PNP_RECORDER._cur else 0
            if n_chunks > last_n_chunks:
                chunk_boundary_actions.append(action_np.copy())
                last_n_chunks = n_chunks
            obs, _, done, _ = env.step(action_np)
            if env.check_success():
                success = True
                break
            if done:
                break
    finally:
        INFERENCE_NUM_STEPS_OVERRIDE = _prev_override

    elapsed = time.time() - t0
    PNP_RECORDER.close_episode(success, step + 1)
    instability = _compute_action_instability(executed_actions, chunk_boundary_actions)

    video_path = None
    should_save = save_video is True or (save_video == 'failures_only' and not success)
    if record_video and should_save and frames and rollout_id is not None:
        os.makedirs(VIDEO_DIR, exist_ok=True)
        video_path = os.path.join(VIDEO_DIR, f'{rollout_id}.mp4')
        imageio.mimsave(video_path, frames, fps=VIDEO_FPS)

    if db is not None:
        db.log_episode(
            rollout_id=rollout_id,
            suite=suite or '',
            task_idx=task_idx or 0,
            task_desc=task_desc,
            episode_idx=episode_idx or 0,
            init_state=init_state,
            success=success,
            n_steps=step + 1,
            elapsed_s=elapsed,
            pnp_cfg=PNP_CONFIG,
            episode_rec=PNP_RECORDER.episodes[-1],
            video_path=video_path,
            method=method,
            final_eval_slice=final_eval_slice,
            num_inference_steps=num_inference_steps,
            num_samples=num_samples,
            instability=instability,
        )

    return success, step + 1, elapsed


def _multi_sample_chunk(policy, batch, base_seed, chunk_idx, num_samples, probe_steps):
    """Sample num_samples chunks; probe U at probe_steps; return lowest-U chunk."""
    saved = (PNP_CONFIG.enabled, PNP_CONFIG.mode, PNP_CONFIG.step_indices, PNP_CONFIG.num_iterations)
    PNP_CONFIG.enabled = True
    PNP_CONFIG.mode = 'uncertainty'
    PNP_CONFIG.step_indices = probe_steps
    PNP_CONFIG.num_iterations = globals().get('PNP_K', 3)
    best_chunk = None
    best_u = float('inf')
    best_chunks = None
    chunk_start = len(PNP_RECORDER._cur['chunks']) if PNP_RECORDER._cur else 0
    for si in range(num_samples):
        policy.reset()  # fresh KV/cache per candidate
        torch.manual_seed(base_seed + chunk_idx * 1000 + si)
        torch.cuda.manual_seed(base_seed + chunk_idx * 1000 + si)
        _pnp_mark_cuda_graph_step()
        with torch.no_grad():
            chunk = policy.predict_action_chunk(batch, noise=None).clone()
        new_chunks = PNP_RECORDER._cur['chunks'][chunk_start:] if PNP_RECORDER._cur else []
        u_vals = [st['u_mean'] for c in new_chunks for st in c.get('steps', [])]
        u_score = float(np.mean(u_vals)) if u_vals else float('inf')
        if u_score < best_u:
            best_u = u_score
            best_chunk = chunk
            best_chunks = list(new_chunks)
        if PNP_RECORDER._cur is not None:
            PNP_RECORDER._cur['chunks'] = PNP_RECORDER._cur['chunks'][:chunk_start]
    if PNP_RECORDER._cur is not None and best_chunks is not None:
        PNP_RECORDER._cur['chunks'].extend(best_chunks)
    PNP_CONFIG.enabled, PNP_CONFIG.mode, PNP_CONFIG.step_indices, PNP_CONFIG.num_iterations = saved
    if best_chunk is None:
        policy.reset()
        _pnp_mark_cuda_graph_step()
        with torch.no_grad():
            return policy.predict_action_chunk(batch, noise=None)
    return best_chunk


def run_episode_multi_sample(env, init_state, policy, task_desc, max_steps, device,
                             ep_meta=None, db=None,
                             suite=None, task_idx=None, episode_idx=None,
                             save_video=False, num_samples=3, probe_steps=(2, 3),
                             method='multi_sample_select', final_eval_slice=0):
    env.reset(); policy.reset()
    obs = env.set_init_state(init_state)
    for _ in range(NUM_STEPS_WAIT):
        obs, _, _, _ = env.step(LIBERO_DUMMY_ACTION)

    _seed = _episode_seed(init_state, episode_idx)
    torch.manual_seed(_seed)
    torch.cuda.manual_seed(_seed)
    PNP_RECORDER.new_episode(ep_meta)
    record_video = save_video in (True, 'failures_only')
    frames = [] if record_video else None
    executed_actions = []
    chunk_boundary_actions = []
    action_queue = []
    chunk_idx = 0

    PNP_CONFIG.enabled = False
    _saved_probe = PNP_CONFIG.step_indices
    PNP_CONFIG.step_indices = probe_steps
    rollout_id = RolloutDB.make_rollout_id(
        suite or '', task_idx or 0, episode_idx or 0, init_state, PNP_CONFIG,
        method=method, num_samples=num_samples) if suite is not None else None
    PNP_CONFIG.step_indices = _saved_probe

    t0 = time.time(); success = False; step = 0
    for step in range(max_steps):
        if record_video:
            frames.append(_agentview_frame(obs))
        if not action_queue:
            raw_obs = obs_to_policy(obs, task_desc, device)
            batch = preprocess(raw_obs)
            chunk = _multi_sample_chunk(policy, batch, _seed, chunk_idx, num_samples, probe_steps)
            chunk_idx += 1
            chunk_np = chunk.squeeze(0).cpu().numpy()
            for i in range(chunk_np.shape[0]):
                action_queue.append(chunk_np[i].copy())
            chunk_boundary_actions.append(action_queue[0].copy())
        action_np = action_queue.pop(0)
        executed_actions.append(action_np.copy())
        obs, _, done, _ = env.step(action_np)
        if env.check_success():
            success = True
            break
        if done:
            break

    elapsed = time.time() - t0
    PNP_RECORDER.close_episode(success, step + 1)
    instability = _compute_action_instability(executed_actions, chunk_boundary_actions)

    video_path = None
    should_save = save_video is True or (save_video == 'failures_only' and not success)
    if record_video and should_save and frames and rollout_id is not None:
        os.makedirs(VIDEO_DIR, exist_ok=True)
        video_path = os.path.join(VIDEO_DIR, f'{rollout_id}.mp4')
        imageio.mimsave(video_path, frames, fps=VIDEO_FPS)

    if db is not None:
        db.log_episode(
            rollout_id=rollout_id,
            suite=suite or '',
            task_idx=task_idx or 0,
            task_desc=task_desc,
            episode_idx=episode_idx or 0,
            init_state=init_state,
            success=success,
            n_steps=step + 1,
            elapsed_s=elapsed,
            pnp_cfg=PNP_CONFIG,
            episode_rec=PNP_RECORDER.episodes[-1],
            video_path=video_path,
            method=method,
            final_eval_slice=final_eval_slice,
            num_inference_steps=None,
            num_samples=num_samples,
            instability=instability,
        )

    return success, step + 1, elapsed


In [9]:
# ── RolloutDB v2: extended schema for controlled final experiment ───────────
import os, sqlite3, hashlib, json as _json, time as _time
import numpy as np

_ADIM = 7
_U_DIM_COLS    = [f'u_d{i}'     for i in range(_ADIM)]
_ASTD_DIM_COLS = [f'a_std_d{i}' for i in range(_ADIM)]
_DIM_COLS      = _U_DIM_COLS + _ASTD_DIM_COLS
_EXTRA_ROLLOUT_COLS = [
    'method', 'final_eval_slice', 'num_inference_steps', 'num_samples',
    'action_delta_l2_mean', 'action_delta_l2_max', 'action_var_mean',
    'gripper_flip_count', 'gripper_flip_rate', 'chunk_disagreement_mean',
]


class RolloutDB:
    """SQLite store for rollout outcomes, P&P uncertainty, and v2 experiment metadata."""

    _DDL = """
    CREATE TABLE IF NOT EXISTS rollouts (
        rollout_id        TEXT PRIMARY KEY,
        suite             TEXT,
        task_idx          INTEGER,
        task_desc         TEXT,
        episode_idx       INTEGER,
        init_state_hash   TEXT,
        success           INTEGER,
        n_steps           INTEGER,
        elapsed_s         REAL,
        pnp_enabled       INTEGER,
        pnp_k             INTEGER,
        pnp_step_indices  TEXT,
        pnp_mode          TEXT,
        u_mean_episode    REAL,
        u_max_episode     REAL,
        n_pnp_activations INTEGER,
        timestamp         TEXT,
        video_path        TEXT,
        method                    TEXT,
        final_eval_slice          INTEGER,
        num_inference_steps       INTEGER,
        num_samples               INTEGER,
        action_delta_l2_mean      REAL,
        action_delta_l2_max       REAL,
        action_var_mean           REAL,
        gripper_flip_count        INTEGER,
        gripper_flip_rate         REAL,
        chunk_disagreement_mean   REAL
    );
    CREATE TABLE IF NOT EXISTS pnp_euler_steps (
        id           INTEGER PRIMARY KEY AUTOINCREMENT,
        rollout_id   TEXT    NOT NULL REFERENCES rollouts(rollout_id),
        chunk_idx    INTEGER NOT NULL,
        euler_step   INTEGER NOT NULL,
        s            REAL,
        u_mean       REAL,
        u_max        REAL,
        a_std_mean   REAL,
        u_d0 REAL, u_d1 REAL, u_d2 REAL, u_d3 REAL, u_d4 REAL, u_d5 REAL, u_d6 REAL,
        a_std_d0 REAL, a_std_d1 REAL, a_std_d2 REAL, a_std_d3 REAL,
        a_std_d4 REAL, a_std_d5 REAL, a_std_d6 REAL
    );
    CREATE INDEX IF NOT EXISTS idx_pes_rollout ON pnp_euler_steps(rollout_id);
    """

    def __init__(self, db_path):
        self.db_path = str(db_path)
        self._con = sqlite3.connect(self.db_path, check_same_thread=False)
        self._con.executescript(self._DDL)
        self._migrate_schema()
        self._con.commit()
        n = self._con.execute('SELECT COUNT(*) FROM rollouts').fetchone()[0]
        print(f"RolloutDB: {self.db_path}  ({n} existing rollouts)")

    def _migrate_schema(self):
        pes_cols = {row[1] for row in self._con.execute("PRAGMA table_info(pnp_euler_steps)")}
        for col in _DIM_COLS:
            if col not in pes_cols:
                self._con.execute(f'ALTER TABLE pnp_euler_steps ADD COLUMN {col} REAL')
        rollout_cols = {row[1] for row in self._con.execute("PRAGMA table_info(rollouts)")}
        type_map = {
            'video_path': 'TEXT', 'method': 'TEXT', 'final_eval_slice': 'INTEGER',
            'num_inference_steps': 'INTEGER', 'num_samples': 'INTEGER',
            'action_delta_l2_mean': 'REAL', 'action_delta_l2_max': 'REAL',
            'action_var_mean': 'REAL', 'gripper_flip_count': 'INTEGER',
            'gripper_flip_rate': 'REAL', 'chunk_disagreement_mean': 'REAL',
        }
        for col, typ in type_map.items():
            if col not in rollout_cols:
                self._con.execute(f'ALTER TABLE rollouts ADD COLUMN {col} {typ}')

    @staticmethod
    def init_state_hash(init_state):
        return hashlib.md5(np.asarray(init_state).tobytes()).hexdigest()[:12]

    @staticmethod
    def make_rollout_id(suite, task_idx, episode_idx, init_state, pnp_cfg,
                        method=None, num_inference_steps=None, num_samples=None):
        cfg_str = _json.dumps({
            'enabled':      pnp_cfg.enabled,
            'k':            pnp_cfg.num_iterations,
            'step_indices': list(pnp_cfg.step_indices) if pnp_cfg.step_indices else None,
            'time_min':     pnp_cfg.time_min,
            'mode':         pnp_cfg.mode,
            'method':       method,
            'num_inference_steps': num_inference_steps,
            'num_samples':  num_samples,
            'probe_steps':  list(pnp_cfg.step_indices) if method == 'multi_sample_select' and pnp_cfg.step_indices else None,
        }, sort_keys=True)
        key = f'{suite}:{task_idx}:{episode_idx}:{RolloutDB.init_state_hash(init_state)}:{cfg_str}'
        return hashlib.sha256(key.encode()).hexdigest()[:16]

    def log_episode(self, rollout_id, suite, task_idx, task_desc, episode_idx,
                    init_state, success, n_steps, elapsed_s, pnp_cfg, episode_rec,
                    video_path=None, method=None, final_eval_slice=0,
                    num_inference_steps=None, num_samples=None, instability=None):
        instability = instability or {}
        all_step_recs = [
            (ci, st)
            for ci, chunk in enumerate(episode_rec.get('chunks', []))
            for st in chunk.get('steps', [])
        ]
        u_vals = [st['u_mean'] for _, st in all_step_recs]
        u_mean_ep = float(np.mean(u_vals)) if u_vals else None
        u_max_ep  = float(np.max(u_vals))  if u_vals else None

        if pnp_cfg.step_indices is not None:
            step_idx_str = _json.dumps(list(pnp_cfg.step_indices))
        else:
            step_idx_str = f'time_min:{pnp_cfg.time_min}' if pnp_cfg.time_min is not None else None

        dim_col_str = ', '.join(_DIM_COLS)
        dim_ph_str  = ', '.join(['?'] * len(_DIM_COLS))

        def _dim_vals(st):
            u_vec     = st.get('u_vec',     [None] * _ADIM)
            a_std_vec = st.get('a_std_vec', [None] * _ADIM)
            return [float(v) if v is not None else None for v in list(u_vec)[:_ADIM]] + \
                   [float(v) if v is not None else None for v in list(a_std_vec)[:_ADIM]]

        with self._con:
            self._con.execute('DELETE FROM pnp_euler_steps WHERE rollout_id = ?', (rollout_id,))
            self._con.execute(
                'INSERT OR REPLACE INTO rollouts VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)',
                (rollout_id, suite, task_idx, task_desc, episode_idx,
                 self.init_state_hash(init_state),
                 int(success), n_steps, round(elapsed_s, 3),
                 int(pnp_cfg.enabled), pnp_cfg.num_iterations,
                 step_idx_str, pnp_cfg.mode,
                 u_mean_ep, u_max_ep, len(all_step_recs),
                 _time.strftime('%Y-%m-%dT%H:%M:%S'), video_path,
                 method, int(final_eval_slice), num_inference_steps, num_samples,
                 instability.get('action_delta_l2_mean'),
                 instability.get('action_delta_l2_max'),
                 instability.get('action_var_mean'),
                 instability.get('gripper_flip_count'),
                 instability.get('gripper_flip_rate'),
                 instability.get('chunk_disagreement_mean')))
            self._con.executemany(
                f'INSERT INTO pnp_euler_steps '
                f'(rollout_id, chunk_idx, euler_step, s, u_mean, u_max, a_std_mean, {dim_col_str}) '
                f'VALUES (?,?,?,?,?,?,?,{dim_ph_str})',
                [(rollout_id, ci, st['step'], st['s'],
                  st['u_mean'], st['u_max'], st['a_std_mean'],
                  *_dim_vals(st))
                 for ci, st in all_step_recs])


    def existing_keys(self, final_eval_slice=1):
        """Set of (suite, task_idx, episode_idx, init_state_hash, method, pnp_step_indices)."""
        rows = self.query(
            'SELECT suite, task_idx, episode_idx, init_state_hash, method, pnp_step_indices '
            'FROM rollouts WHERE final_eval_slice = ?',
            (int(final_eval_slice),),
        )
        return {
            (r['suite'], r['task_idx'], r['episode_idx'], r['init_state_hash'],
             r.get('method'), r.get('pnp_step_indices'))
            for r in rows
        }

    def import_v1_rollouts(self, v1_db_path, episode_keys, *,
                           v1_video_dir=None, v2_video_dir=None,
                           step_configs=None, base_inference_steps=10,
                           dry_run=False):
        """Copy reusable v1 rows into v2 DB with method labels.

        episode_keys: set of (suite, task_idx, episode_idx, init_state_hash)
        Imports:
          v1 pnp_enabled=0  -> method='vanilla'
          v1 pnp_mode='uncertainty' -> method='pnp_uncertainty_only'
          v1 pnp_mode='both'        -> method='pnp_refinement'
        Only imports step configs in step_configs (JSON strings) for P&P rows.
        """
        import shutil
        step_configs = step_configs or []
        allowed_steps = set()
        for cfg in step_configs:
            allowed_steps.add(_json.dumps(list(cfg)))
            allowed_steps.add(str(list(cfg)).replace(' ', ''))  # tolerate loose formatting

        v1 = sqlite3.connect(v1_db_path)
        v1.row_factory = sqlite3.Row
        imported = {'vanilla': 0, 'pnp_uncertainty_only': 0, 'pnp_refinement': 0, 'out_of_slice': 0, 'wrong_step_config': 0}

        v1_rollout_cols = {r[1] for r in v1.execute('PRAGMA table_info(rollouts)')}
        v2_rollout_cols = {r[1] for r in self._con.execute('PRAGMA table_info(rollouts)')}

        rows = v1.execute('SELECT * FROM rollouts').fetchall()
        for row in rows:
            row = dict(row)
            key = (row['suite'], row['task_idx'], row['episode_idx'], row['init_state_hash'])
            if key not in episode_keys:
                imported['out_of_slice'] += 1
                continue

            if row.get('pnp_enabled', 0) == 0:
                method = 'vanilla'
                pnp_mode = row.get('pnp_mode')
                step_idx_str = None
                num_steps = base_inference_steps
                num_samples = None
            else:
                step_idx_str = row.get('pnp_step_indices')
                if step_configs and step_idx_str not in allowed_steps:
                    imported['wrong_step_config'] += 1
                    continue
                mode = row.get('pnp_mode')
                if mode == 'uncertainty':
                    method = 'pnp_uncertainty_only'
                elif mode == 'both':
                    method = 'pnp_refinement'
                else:
                    imported['wrong_step_config'] += 1
                    continue
                pnp_mode = mode
                num_steps = None
                num_samples = None

            # Build v2 rollout_id
            class _Cfg:
                pass
            cfg = _Cfg()
            cfg.enabled = bool(row.get('pnp_enabled', 0))
            cfg.num_iterations = row.get('pnp_k') or 3
            if step_idx_str:
                try:
                    cfg.step_indices = tuple(_json.loads(step_idx_str))
                except Exception:
                    cfg.step_indices = None
            else:
                cfg.step_indices = None
            cfg.time_min = None
            cfg.mode = pnp_mode or 'both'

            cfg_str = _json.dumps({
                'enabled': cfg.enabled, 'k': cfg.num_iterations,
                'step_indices': list(cfg.step_indices) if cfg.step_indices else None,
                'time_min': cfg.time_min, 'mode': cfg.mode,
                'method': method, 'num_inference_steps': num_steps, 'num_samples': num_samples,
                'probe_steps': None,
            }, sort_keys=True)
            new_id = hashlib.sha256(
                f"{row['suite']}:{row['task_idx']}:{row['episode_idx']}:{row['init_state_hash']}:{cfg_str}".encode()
            ).hexdigest()[:16]

            if dry_run:
                imported[method] += 1
                continue

            video_path = None
            if v1_video_dir and v2_video_dir:
                v1_vid = os.path.join(v1_video_dir, f"{row['rollout_id']}.mp4")
                if os.path.isfile(v1_vid):
                    os.makedirs(v2_video_dir, exist_ok=True)
                    v2_vid = os.path.join(v2_video_dir, f'{new_id}.mp4')
                    if not os.path.isfile(v2_vid):
                        shutil.copy2(v1_vid, v2_vid)
                    video_path = v2_vid
                elif row.get('video_path') and os.path.isfile(str(row['video_path'])):
                    video_path = str(row['video_path'])

            dest = {c: row.get(c) for c in [
                'suite', 'task_idx', 'task_desc', 'episode_idx', 'init_state_hash',
                'success', 'n_steps', 'elapsed_s', 'pnp_enabled', 'pnp_k',
                'pnp_mode', 'u_mean_episode', 'u_max_episode',
                'n_pnp_activations', 'timestamp',
            ] if c in row}
            dest['pnp_step_indices'] = step_idx_str
            dest.update({
                'rollout_id': new_id,
                'video_path': video_path,
                'method': method,
                'final_eval_slice': 1,
                'num_inference_steps': num_steps,
                'num_samples': num_samples,
                'action_delta_l2_mean': None,
                'action_delta_l2_max': None,
                'action_var_mean': None,
                'gripper_flip_count': None,
                'gripper_flip_rate': None,
                'chunk_disagreement_mean': None,
            })

            cols = [c for c in dest if c in v2_rollout_cols]
            placeholders = ','.join(['?'] * len(cols))
            col_names = ','.join(cols)
            with self._con:
                self._con.execute(f'DELETE FROM pnp_euler_steps WHERE rollout_id = ?', (new_id,))
                self._con.execute(
                    f'INSERT OR REPLACE INTO rollouts ({col_names}) VALUES ({placeholders})',
                    [dest[c] for c in cols],
                )
                pes_rows = v1.execute(
                    'SELECT * FROM pnp_euler_steps WHERE rollout_id = ?', (row['rollout_id'],)
                ).fetchall()
                if pes_rows:
                    pes_cols = [d[1] for d in v1.execute('PRAGMA table_info(pnp_euler_steps)')]
                    pes_cols_v2 = [d[1] for d in self._con.execute('PRAGMA table_info(pnp_euler_steps)')]
                    insert_cols = [c for c in pes_cols if c in pes_cols_v2 and c != 'id']
                    ph = ','.join(['?'] * len(insert_cols))
                    col_str = ','.join(insert_cols)
                    for pr in pes_rows:
                        prd = dict(pr)
                        prd['rollout_id'] = new_id
                        self._con.execute(
                            f'INSERT INTO pnp_euler_steps ({col_str}) VALUES ({ph})',
                            [prd.get(c) for c in insert_cols],
                        )
            imported[method] += 1

        v1.close()
        self._con.commit()
        return imported


    def close(self):
        self._con.close()

    def query(self, sql, params=()):
        cur = self._con.execute(sql, params)
        cols = [d[0] for d in cur.description]
        return [dict(zip(cols, row)) for row in cur.fetchall()]

    def summary(self):
        rows = self.query("""
            SELECT suite, task_idx, method,
                   COUNT(*) AS n_ep,
                   ROUND(AVG(success)*100, 1) AS sr_pct,
                   ROUND(AVG(u_mean_episode), 5) AS u_mean_all
            FROM rollouts
            GROUP BY suite, task_idx, method
            ORDER BY suite, task_idx, method
        """)
        print(f"{'suite':<18} {'task':>4} {'method':<22} {'n':>4} {'sr%':>6} {'u_all':>10}")
        print('-' * 72)
        for r in rows:
            print(f"{r['suite']:<18} {r['task_idx']:>4} {str(r.get('method','')):<22} {r['n_ep']:>4} {r['sr_pct']:>6} {str(r['u_mean_all']):>10}")
        return rows


_results_root = RESULTS_DIR if 'RESULTS_DIR' in dir() else '/content/drive/MyDrive/cs159-sp26/results_v2'
DB_PATH = os.path.join(_results_root, 'rollouts_v2.db')
VIDEO_DIR = os.path.join(_results_root, 'videos_v2')
os.makedirs(VIDEO_DIR, exist_ok=True)
DB = RolloutDB(DB_PATH)
print(f"Use DB= parameter in run_episode_pnp() to auto-log, or call DB.log_episode() manually.")
print(f"Failure videos: {VIDEO_DIR}  (save_video='failures_only')")


RolloutDB: /content/drive/MyDrive/cs159-sp26/results_v2/rollouts_v2.db  (0 existing rollouts)
Use DB= parameter in run_episode_pnp() to auto-log, or call DB.log_episode() manually.
Failure videos: /content/drive/MyDrive/cs159-sp26/results_v2/videos_v2  (save_video='failures_only')


---
## Section 6: Final controlled experiment (v2)

1. Derive slice from v1 DB
2. Import v1 rows
3. Run missing methods


In [10]:
# === FINAL EXPERIMENT v2: derive fixed evaluation slice from v1 DB ===
import sqlite3
from collections import defaultdict

V1_DB = f'{SHARED}/results/rollouts.db'
V1_VIDEO_DIR = f'{SHARED}/results/videos'
FINAL_SUITES = ['libero_goal', 'libero_spatial']
FINAL_STEP_CONFIGS = [(2, 3), (3, 4), (4, 5)]
FINAL_MAX_TASKS = 8
FINAL_EPISODE_IDXS = list(range(10))
PNP_K = 3
BASELINE_STEPS = 10
EXTRA_STEPS_MATCHED = BASELINE_STEPS + PNP_K * 2  # matched to 2-step P&P with K=3
MULTI_SAMPLE_N = 3
MULTI_SAMPLE_PROBE_STEPS = (2, 3)

# v1 reuse controls
IMPORT_V1 = True          # import v1 vanilla + P&P rows before running new methods
SKIP_COMPLETED = True     # skip episode×method combos already in v2 DB

METHODS = ['vanilla', 'extra_steps', 'multi_sample_select',
           'pnp_uncertainty_only', 'pnp_refinement']
# After v1 import, vanilla + pnp_* are usually done — run only new baselines by default:
RUN_METHODS = ['extra_steps', 'multi_sample_select']
# Full matrix (incl. re-run): RUN_METHODS = METHODS

# ── Derive task list from v1 Phase-1 baseline failures (read-only) ─────────────
task_fail_counts = defaultdict(int)
if os.path.isfile(V1_DB):
    _v1 = sqlite3.connect(V1_DB)
    rows = _v1.execute(
        "SELECT suite, task_idx, COUNT(*) as n_fail "
        "FROM rollouts WHERE pnp_enabled=0 AND success=0 AND suite IN (?,?) "
        "GROUP BY suite, task_idx ORDER BY n_fail DESC",
        tuple(FINAL_SUITES),
    ).fetchall()
    _v1.close()
    for suite, task_idx, n_fail in rows:
        task_fail_counts[(suite, task_idx)] = n_fail
    print(f'Read v1 DB: {V1_DB}  ({len(rows)} failing tasks in {FINAL_SUITES})')
else:
    print(f'v1 DB not found at {V1_DB} — using fallback task list')
    task_fail_counts = {
        ('libero_spatial', 4): 9, ('libero_spatial', 5): 8, ('libero_spatial', 8): 4,
        ('libero_goal', 3): 3, ('libero_goal', 5): 4, ('libero_goal', 6): 7,
    }

ranked_tasks = sorted(task_fail_counts.keys(), key=lambda k: -task_fail_counts[k])[:FINAL_MAX_TASKS]
print(f'Selected {len(ranked_tasks)} tasks: {ranked_tasks}')

benchmark_dict = benchmark.get_benchmark_dict()
FINAL_EPISODES = []
EPISODE_KEYS = set()
for suite, task_idx in ranked_tasks:
    task_suite = benchmark_dict[suite]()
    task = task_suite.get_task(task_idx)
    init_states = task_suite.get_task_init_states(task_idx)
    bddl_path = os.path.join(get_libero_path('bddl_files'), task.problem_folder, task.bddl_file)
    max_steps = MAX_STEPS_MAP.get(suite, 300)
    for ep_idx in FINAL_EPISODE_IDXS:
        if ep_idx >= len(init_states):
            continue
        init_state = init_states[ep_idx]
        ish = RolloutDB.init_state_hash(init_state)
        EPISODE_KEYS.add((suite, task_idx, ep_idx, ish))
        FINAL_EPISODES.append(dict(
            suite=suite, task_idx=task_idx, task_desc=task.language,
            ep_idx=ep_idx, init_state=init_state, bddl_path=bddl_path,
            max_steps=max_steps, init_state_hash=ish,
        ))

print(f'FINAL_EPISODES: {len(FINAL_EPISODES)} episodes across {len(ranked_tasks)} tasks')


Read v1 DB: /content/drive/MyDrive/cs159-sp26/results/rollouts.db  (13 failing tasks in ['libero_goal', 'libero_spatial'])
Selected 8 tasks: [('libero_spatial', 5), ('libero_goal', 2), ('libero_goal', 3), ('libero_spatial', 8), ('libero_goal', 6), ('libero_goal', 0), ('libero_goal', 1), ('libero_goal', 5)]
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
FINAL_EPISODES: 80 episodes across 8 tasks


In [11]:
# === FINAL EXPERIMENT v2: import reusable v1 results ===
if not IMPORT_V1:
    print('IMPORT_V1=False — skipping v1 import.')
elif not os.path.isfile(V1_DB):
    print(f'v1 DB not found at {V1_DB} — run test_pi05_jennifer.ipynb Phase 1 first, or set IMPORT_V1=False.')
elif not EPISODE_KEYS:
    print('No EPISODE_KEYS — run slice derivation cell first.')
else:
    stats = DB.import_v1_rollouts(
        V1_DB,
        EPISODE_KEYS,
        v1_video_dir=V1_VIDEO_DIR,
        v2_video_dir=VIDEO_DIR,
        step_configs=FINAL_STEP_CONFIGS,
        base_inference_steps=BASELINE_STEPS,
    )
    print('Imported from v1 into v2 DB:')
    for k, v in stats.items():
        print(f'  {k}: {v}')
    print('Existing v2 keys:', len(DB.existing_keys()))
    DB.summary()



Imported from v1 into v2 DB:
  vanilla: 80
  pnp_uncertainty_only: 57
  pnp_refinement: 57
  out_of_slice: 620
  wrong_step_config: 76
Existing v2 keys: 194
suite              task method                    n    sr%      u_all
------------------------------------------------------------------------
libero_goal           0 pnp_refinement            3  100.0    0.03454
libero_goal           0 pnp_uncertainty_only      3   66.7    0.04504
libero_goal           0 vanilla                  10   90.0       None
libero_goal           1 pnp_refinement            3  100.0    0.01523
libero_goal           1 pnp_uncertainty_only      3   66.7    0.02529
libero_goal           1 vanilla                  10   90.0       None
libero_goal           2 pnp_refinement            9   77.8    0.03157
libero_goal           2 pnp_uncertainty_only      9   88.9    0.03068
libero_goal           2 vanilla                  10   70.0       None
libero_goal           3 pnp_refinement            9   88.9    0.02242


In [12]:
# === FINAL EXPERIMENT v2: matched method matrix on fixed slice ===
import time, json as _json
from itertools import groupby
from tqdm.notebook import tqdm

if not FINAL_EPISODES:
    print('No FINAL_EPISODES — run the slice derivation cell first.')
elif not RUN_METHODS:
    print('RUN_METHODS is empty — nothing to execute. Run import cell; set RUN_METHODS if needed.')
    DB.summary()
else:
    sorted_eps = sorted(FINAL_EPISODES, key=lambda x: (x['suite'], x['task_idx']))
    total_runs = 0
    matrix_start = time.time()

    for method in RUN_METHODS:
        print(f'\n{"#"*60}\nMethod: {method}  |  {len(FINAL_EPISODES)} episodes\n{"#"*60}')
        PNP_CONFIG.enabled = False
        INFERENCE_NUM_STEPS_OVERRIDE = None

        if method == 'vanilla':
            configs = [None]
        elif method == 'extra_steps':
            configs = [None]
            INFERENCE_NUM_STEPS_OVERRIDE = EXTRA_STEPS_MATCHED
        elif method == 'multi_sample_select':
            configs = [None]
        elif method in ('pnp_uncertainty_only', 'pnp_refinement'):
            configs = list(FINAL_STEP_CONFIGS)
        else:
            configs = [None]

        for step_indices in configs:
            steps_label = str(list(step_indices)) if step_indices else 'none'
            if method == 'pnp_uncertainty_only':
                PNP_CONFIG.enabled = True
                PNP_CONFIG.mode = 'uncertainty'
                PNP_CONFIG.step_indices = step_indices
                PNP_CONFIG.num_iterations = PNP_K
            elif method == 'pnp_refinement':
                PNP_CONFIG.enabled = True
                PNP_CONFIG.mode = 'both'
                PNP_CONFIG.step_indices = step_indices
                PNP_CONFIG.num_iterations = PNP_K
            elif method in ('vanilla', 'extra_steps', 'multi_sample_select'):
                PNP_CONFIG.enabled = False
                PNP_CONFIG.mode = 'both'
                PNP_CONFIG.step_indices = None

            PNP_CONFIG.time_min = None
            PNP_RECORDER.reset()
            method_results = []
            n_skipped = 0
            step_key = _json.dumps(list(step_indices)) if step_indices else None
            completed_keys = DB.existing_keys() if SKIP_COMPLETED else set()

            for (suite, task_idx), group_iter in groupby(
                    sorted_eps, key=lambda x: (x['suite'], x['task_idx'])):
                episodes = list(group_iter)
                env = OffScreenRenderEnv(
                    bddl_file_name=episodes[0]['bddl_path'], camera_names=CAMERAS,
                    camera_heights=IMG_SIZE, camera_widths=IMG_SIZE,
                    has_offscreen_renderer=True, use_camera_obs=True,
                    has_renderer=False, reward_shaping=False,
                )
                desc = f'  {method} steps={steps_label} T{task_idx+1}'
                ep_bar = tqdm(episodes, desc=desc, leave=False)
                try:
                    for ep_info in ep_bar:
                        ep_key = (
                            suite, task_idx, ep_info['ep_idx'], ep_info['init_state_hash'],
                            method, step_key,
                        )
                        if SKIP_COMPLETED and ep_key in completed_keys:
                            n_skipped += 1
                            ep_bar.set_postfix(status='skip')
                            continue
                        common = dict(
                            env=env, init_state=ep_info['init_state'], policy=policy,
                            task_desc=ep_info['task_desc'], max_steps=ep_info['max_steps'],
                            device=device,
                            ep_meta={'suite': suite, 'task_idx': task_idx,
                                     'episode': ep_info['ep_idx'], 'method': method,
                                     'step_indices': list(step_indices) if step_indices else None},
                            db=DB, suite=suite, task_idx=task_idx,
                            episode_idx=ep_info['ep_idx'],
                            save_video='failures_only',
                            method=method, final_eval_slice=1,
                        )
                        if method == 'multi_sample_select':
                            success, n_steps, elapsed = run_episode_multi_sample(
                                **common, num_samples=MULTI_SAMPLE_N,
                                probe_steps=MULTI_SAMPLE_PROBE_STEPS)
                        elif method == 'extra_steps':
                            success, n_steps, elapsed = run_episode_pnp(
                                **common, num_inference_steps=EXTRA_STEPS_MATCHED)
                        elif method == 'vanilla':
                            success, n_steps, elapsed = run_episode_pnp(
                                **common, num_inference_steps=BASELINE_STEPS)
                        else:
                            success, n_steps, elapsed = run_episode_pnp(**common)

                        method_results.append(dict(success=success, n_steps=n_steps))
                        total_runs += 1
                        ep_bar.set_postfix(status='\u2713' if success else '\u2717')
                finally:
                    env.close()

            if method_results:
                sr = sum(r['success'] for r in method_results) / len(method_results)
                print(f'  {method} steps={steps_label} SR: {sr:.1%}  ({len(method_results)} ran, {n_skipped} skipped)')
            else:
                print(f'  {method} steps={steps_label}: all {n_skipped} episodes skipped (already in DB)')

    INFERENCE_NUM_STEPS_OVERRIDE = None
    PNP_CONFIG.enabled = False
    elapsed_min = (time.time() - matrix_start) / 60
    print(f'\nDone. {total_runs} rollouts in {elapsed_min:.1f} min. DB summary:')
    DB.summary()



############################################################
Method: extra_steps  |  80 episodes
############################################################


  extra_steps steps=none T1:   0%|          | 0/10 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  extra_steps steps=none T2:   0%|          | 0/10 [00:00<?, ?it/s]

  extra_steps steps=none T3:   0%|          | 0/10 [00:00<?, ?it/s]

...

  multi_sample_select steps=none T6:   0%|          | 0/10 [00:00<?, ?it/s]

  multi_sample_select steps=none T7:   0%|          | 0/10 [00:00<?, ?it/s]

  multi_sample_select steps=none T6:   0%|          | 0/10 [00:00<?, ?it/s]

  multi_sample_select steps=none T9:   0%|          | 0/10 [00:00<?, ?it/s]

  multi_sample_select steps=none SR: 0.0%  (80 ran, 0 skipped)

Done. 160 rollouts in 192.5 min. DB summary:
suite              task method                    n    sr%      u_all
------------------------------------------------------------------------
libero_goal           0 extra_steps              10  100.0       None
libero_goal           0 multi_sample_select      10    0.0    0.08964
libero_goal           0 pnp_refinement            3  100.0    0.03454
libero_goal           0 pnp_uncertainty_only      3   66.7    0.04504
libero_goal           0 vanilla                  10   90.0       None
libero_goal           1 extra_steps              10  100.0       None
libero_goal           1 multi_sample_select      10    0.0    0.04635
libero_goal           1 pnp_refinement            3  100.0    0.01523
libero_goal           1 pnp_uncertainty_only      3   66.7    0.02529
libero_goal           1 vanilla                  10   90.0       None
libero_goal           2 extra_steps             

In [13]:
# === Finish v2 P&P matrix: run missing pnp_uncertainty_only + pnp_refinement ===
# Skips the ~19 episodes already imported from v1; runs the remaining ~61.
# Expect ~61 eps × 2 methods × 3 step configs ≈ 366 rollouts.

import time, json as _json
from itertools import groupby
from tqdm.notebook import tqdm

RUN_METHODS_FINISH = ['pnp_uncertainty_only', 'pnp_refinement']
SKIP_COMPLETED_FINISH = True  # keep v1-imported rows; only fill gaps

if not FINAL_EPISODES:
    raise RuntimeError('Run the slice derivation cell first (FINAL_EPISODES is empty).')
if 'DB' not in globals() or DB is None:
    raise RuntimeError('Run Section 5 first (DB not defined).')
if 'policy' not in globals() or policy is None:
    raise RuntimeError('Run Section 4 first (policy not loaded).')

completed_keys = DB.existing_keys() if SKIP_COMPLETED_FINISH else set()
missing = []
for ep in FINAL_EPISODES:
    for method in RUN_METHODS_FINISH:
        for step_indices in FINAL_STEP_CONFIGS:
            step_key = _json.dumps(list(step_indices))
            ep_key = (ep['suite'], ep['task_idx'], ep['ep_idx'], ep['init_state_hash'], method, step_key)
            if ep_key not in completed_keys:
                missing.append(ep_key)

print(f'FINAL_EPISODES: {len(FINAL_EPISODES)}')
print(f'Existing v2 keys: {len(completed_keys)}')
print(f'Missing P&P rollouts to run: {len(missing)}')
print(f'Methods: {RUN_METHODS_FINISH}')
print(f'Step configs: {FINAL_STEP_CONFIGS}')
if not missing:
    print('Nothing to do — P&P matrix already complete.')
    DB.summary()
else:
    sorted_eps = sorted(FINAL_EPISODES, key=lambda x: (x['suite'], x['task_idx']))
    total_runs = 0
    matrix_start = time.time()

    for method in RUN_METHODS_FINISH:
        print(f'\n{"#"*60}\nMethod: {method}  |  {len(FINAL_EPISODES)} episodes\n{"#"*60}')
        PNP_CONFIG.enabled = False
        INFERENCE_NUM_STEPS_OVERRIDE = None
        configs = list(FINAL_STEP_CONFIGS)

        for step_indices in configs:
            steps_label = str(list(step_indices))
            if method == 'pnp_uncertainty_only':
                PNP_CONFIG.enabled = True
                PNP_CONFIG.mode = 'uncertainty'
                PNP_CONFIG.step_indices = step_indices
                PNP_CONFIG.num_iterations = PNP_K
            else:  # pnp_refinement
                PNP_CONFIG.enabled = True
                PNP_CONFIG.mode = 'both'
                PNP_CONFIG.step_indices = step_indices
                PNP_CONFIG.num_iterations = PNP_K

            PNP_CONFIG.time_min = None
            PNP_RECORDER.reset()
            method_results = []
            n_skipped = 0
            step_key = _json.dumps(list(step_indices))

            for (suite, task_idx), group_iter in groupby(
                    sorted_eps, key=lambda x: (x['suite'], x['task_idx'])):
                episodes = list(group_iter)
                env = OffScreenRenderEnv(
                    bddl_file_name=episodes[0]['bddl_path'], camera_names=CAMERAS,
                    camera_heights=IMG_SIZE, camera_widths=IMG_SIZE,
                    has_offscreen_renderer=True, use_camera_obs=True,
                    has_renderer=False, reward_shaping=False,
                )
                desc = f'  {method} steps={steps_label} T{task_idx+1}'
                ep_bar = tqdm(episodes, desc=desc, leave=False)
                try:
                    for ep_info in ep_bar:
                        ep_key = (
                            suite, task_idx, ep_info['ep_idx'], ep_info['init_state_hash'],
                            method, step_key,
                        )
                        if SKIP_COMPLETED_FINISH and ep_key in completed_keys:
                            n_skipped += 1
                            ep_bar.set_postfix(status='skip')
                            continue

                        common = dict(
                            env=env, init_state=ep_info['init_state'], policy=policy,
                            task_desc=ep_info['task_desc'], max_steps=ep_info['max_steps'],
                            device=device,
                            ep_meta={'suite': suite, 'task_idx': task_idx,
                                     'episode': ep_info['ep_idx'], 'method': method,
                                     'step_indices': list(step_indices)},
                            db=DB, suite=suite, task_idx=task_idx,
                            episode_idx=ep_info['ep_idx'],
                            save_video='failures_only',
                            method=method, final_eval_slice=1,
                        )
                        success, n_steps, elapsed = run_episode_pnp(**common)

                        method_results.append(dict(success=success, n_steps=n_steps))
                        total_runs += 1
                        completed_keys.add(ep_key)  # avoid duplicate if cell re-run mid-way
                        ep_bar.set_postfix(status='✓' if success else '✗')
                finally:
                    env.close()

            if method_results:
                sr = sum(r['success'] for r in method_results) / len(method_results)
                print(f'  {method} steps={steps_label} SR: {sr:.1%}  '
                      f'({len(method_results)} ran, {n_skipped} skipped)')
            else:
                print(f'  {method} steps={steps_label}: all {n_skipped} skipped (already in DB)')

    INFERENCE_NUM_STEPS_OVERRIDE = None
    PNP_CONFIG.enabled = False
    elapsed_min = (time.time() - matrix_start) / 60
    print(f'\nDone. {total_runs} new rollouts in {elapsed_min:.1f} min.')
    print('DB summary:')
    DB.summary()

FINAL_EPISODES: 80
Existing v2 keys: 354
Missing P&P rollouts to run: 366
Methods: ['pnp_uncertainty_only', 'pnp_refinement']
Step configs: [(2, 3), (3, 4), (4, 5)]

############################################################
Method: pnp_uncertainty_only  |  80 episodes
############################################################


  pnp_uncertainty_only steps=[2, 3] T1:   0%|          | 0/10 [00:00<?, ?it/s]

  pnp_uncertainty_only steps=[2, 3] T2:   0%|          | 0/10 [00:00<?, ?it/s]

  pnp_uncertainty_only steps=[2, 3] T3:   0%|          | 0/10 [00:00<?, ?it/s]

  pnp_uncertainty_only steps=[2, 3] T4:   0%|          | 0/10 [00:00<?, ?it/s]

...

  pnp_refinement steps=[4, 5] T6:   0%|          | 0/10 [00:00<?, ?it/s]

  pnp_refinement steps=[4, 5] T7:   0%|          | 0/10 [00:00<?, ?it/s]

  pnp_refinement steps=[4, 5] T6:   0%|          | 0/10 [00:00<?, ?it/s]

  pnp_refinement steps=[4, 5] T9:   0%|          | 0/10 [00:00<?, ?it/s]

  pnp_refinement steps=[4, 5] SR: 90.2%  (61 ran, 19 skipped)

Done. 366 new rollouts in 288.5 min.
DB summary:
suite              task method                    n    sr%      u_all
------------------------------------------------------------------------
libero_goal           0 extra_steps              10  100.0       None
libero_goal           0 multi_sample_select      10    0.0    0.08964
libero_goal           0 pnp_refinement           30  100.0    0.03166
libero_goal           0 pnp_uncertainty_only     30   90.0    0.03708
libero_goal           0 vanilla                  10   90.0       None
libero_goal           1 extra_steps              10  100.0       None
libero_goal           1 multi_sample_select      10    0.0    0.04635
libero_goal           1 pnp_refinement           30  100.0    0.01781
libero_goal           1 pnp_uncertainty_only     30   93.3     0.0196
libero_goal           1 vanilla                  10   90.0       None
libero_goal           2 extra_steps          

In [22]:
# === Smoke test: one multi_sample_select episode (no DB write) ===
# Prereqs: Sections 1–5, slice derivation. Run Section 5 for fixed run_episode_multi_sample.
# Pick an episode vanilla succeeded on — if multi_sample also succeeds, the fix likely works.

SMOKE_EP_IDX = 0  # index into FINAL_EPISODES; 0 = first slice episode (libero_goal task 0 ep 0)

for name in ('FINAL_EPISODES', 'policy', 'DB'):
    if name not in globals() or globals()[name] is None:
        raise RuntimeError(f'Run prerequisite cells first ({name} missing).')

ep = FINAL_EPISODES[SMOKE_EP_IDX]
probe_steps = MULTI_SAMPLE_PROBE_STEPS if 'MULTI_SAMPLE_PROBE_STEPS' in globals() else (2, 3)
num_samples = MULTI_SAMPLE_N if 'MULTI_SAMPLE_N' in globals() else 3

# Vanilla outcome on same episode (from DB, if present)
van_row = DB._con.execute(
    'SELECT success, n_steps FROM rollouts WHERE method=? AND final_eval_slice=1 '
    'AND suite=? AND task_idx=? AND episode_idx=? AND init_state_hash=?',
    ('vanilla', ep['suite'], ep['task_idx'], ep['ep_idx'], ep['init_state_hash']),
).fetchone()
if van_row:
    print(f"Vanilla on this ep: success={bool(van_row[0])}  n_steps={van_row[1]}")
else:
    print('No vanilla row for this episode in DB (smoke test still runs).')

print(f"Smoke test: {ep['suite']} task {ep['task_idx']} ep {ep['ep_idx']}")
print(f"  {ep['task_desc'][:60]}...")

env = OffScreenRenderEnv(
    bddl_file_name=ep['bddl_path'], camera_names=CAMERAS,
    camera_heights=IMG_SIZE, camera_widths=IMG_SIZE,
    has_offscreen_renderer=True, use_camera_obs=True,
    has_renderer=False, reward_shaping=False,
)
PNP_CONFIG.enabled = False
INFERENCE_NUM_STEPS_OVERRIDE = None
try:
    t0 = time.time()
    success, n_steps, elapsed = run_episode_multi_sample(
        env=env, init_state=ep['init_state'], policy=policy,
        task_desc=ep['task_desc'], max_steps=ep['max_steps'], device=device,
        ep_meta={'suite': ep['suite'], 'task_idx': ep['task_idx'],
                 'episode': ep['ep_idx'], 'method': 'multi_sample_select'},
        db=None,  # do not write — smoke test only
        suite=ep['suite'], task_idx=ep['task_idx'], episode_idx=ep['ep_idx'],
        save_video=False,
        num_samples=num_samples, probe_steps=probe_steps,
    )
finally:
    env.close()

print(f"\nmulti_sample_select: success={success}  n_steps={n_steps}  elapsed={elapsed:.1f}s")
if n_steps >= ep['max_steps'] and not success:
    print('  → Timed out at max_steps (same failure mode as broken 0% runs).')
elif success:
    print('  → SUCCESS — fixed implementation looks plausible; safe to run full 80-ep cell.')
elif van_row and van_row[0]:
    print('  → Failed but vanilla succeeded here — may be variance; try 2–3 more eps before full run.')
else:
    print('  → Failed; vanilla also failed here — pick SMOKE_EP_IDX with vanilla success for a sharper test.')


Vanilla on this ep: success=True  n_steps=89
Smoke test: libero_spatial task 5 ep 0
  pick up the black bowl on the ramekin and place it on the pl...

multi_sample_select: success=False  n_steps=220  elapsed=73.9s
  → Timed out at max_steps (same failure mode as broken 0% runs).


In [20]:
DRIVE_DB_PATH = f'{SHARED}/results_v2/rollouts_v2.db'

if hasattr(DB, 'sync_to_path'):
    DB.sync_to_path(DRIVE_DB_PATH)
    DB.verify_disk(DRIVE_DB_PATH)
else:
    import sqlite3, shutil
    tmp = '/content/rollouts_v2_flush.db'
    dst = sqlite3.connect(tmp)
    DB._con.backup(dst)
    dst.close()
    shutil.copy2(tmp, DRIVE_DB_PATH)
    mem = DB._con.execute(
        'SELECT COUNT(*) FROM rollouts WHERE final_eval_slice=1').fetchone()[0]
    disk = sqlite3.connect(DRIVE_DB_PATH).execute(
        'SELECT COUNT(*) FROM rollouts WHERE final_eval_slice=1').fetchone()[0]
    print(f'connection={mem}  disk={disk}')

connection=650  disk=650
